<a href="https://colab.research.google.com/github/jaloaiza/genaiassignments/blob/main/assignments/FinalProject/CreateHFDataset.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Create Hugging Face Dataset

<a target="_blank" href="https://colab.research.google.com/github/simonguest/CS-394/blob/main/src/07/notebooks/create-dataset.ipynb">
  <img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/>
</a>
<a target="_blank" href="https://github.com/simonguest/CS-394/raw/refs/heads/main/src/07/notebooks/create-dataset.ipynb">
  <img src="https://img.shields.io/badge/Download_.ipynb-blue" alt="Download .ipynb"/>
</a>

## Install dependencies

In [ ]:
!uv pip install datasets

Using Python 3.12.13 environment at: /usr
Checked 1 package in 86ms


## Configuration

**Note**: If you are running this notebook on Colab, be sure to first upload your training data (.jsonl) files.

In [ ]:
TRAIN_FILE = "./data/train.jsonl"
VALIDATION_FILE = "./data/validation.jsonl"
TEST_FILE = "./data/test.jsonl"

DATASET_REPO = "jujuloaiza/profilerchatbot"

## Create dataset functions

In [ ]:
from datasets import Dataset, DatasetDict
import json


def load_jsonl(file_path):
    data = []
    with open(file_path, "r", encoding="utf-8") as f:
        for line_num, line in enumerate(f, 1):
            line = line.strip()
            if not line:  # Skip empty lines
                continue
            try:
                # Try parsing the line
                data.append(json.loads(line))
            except json.JSONDecodeError as e:
                print(f"Warning: Error parsing line {line_num}: {e}")
                print(f"Problematic line: {line[:200]}...")
    return data


def create_hf_dataset(train_file, val_file, test_file):
    # Load the data
    train_data = load_jsonl(train_file)
    val_data = load_jsonl(val_file)
    test_data = load_jsonl(test_file)

    # Create datasets
    train_dataset = Dataset.from_list(train_data)
    val_dataset = Dataset.from_list(val_data)
    test_dataset = Dataset.from_list(test_data)

    # Combine into DatasetDict
    dataset_dict = DatasetDict(
        {
            "train": train_dataset,
            "validation": val_dataset,
            "test": test_dataset,
        }
    )

    return dataset_dict


# Create and validate the dataset is ready to upload
dataset = create_hf_dataset(TRAIN_FILE, VALIDATION_FILE, TEST_FILE)
print(f"Train samples: {len(dataset['train'])}")
print(f"Validation samples: {len(dataset['validation'])}")
print(f"Test samples: {len(dataset['test'])}")
print(f"\nSample entry: {dataset['train'][0]}")

Train samples: 500
Validation samples: 50
Test samples: 10

Sample entry: {'messages': [{'content': 'gr1ndyL00tL0l', 'role': 'user'}], 'labels': {'communication_style': 'sarcastic', 'player_type': 'troll_light', 'verbosity': 'one_word', 'writing_pattern': 'heavy_typos'}}


## Get Hugging Face token

In [ ]:
import sys
import os
from dotenv import load_dotenv

if 'google.colab' in sys.modules:
  from google.colab import userdata # type:ignore
  os.environ["WANDB_API_KEY"] = userdata.get("WANDB_API_KEY")
else:
  load_dotenv()
  print("Loaded env vars from .env")

## Upload to Hugging Face

In [ ]:
def upload_dataset(dataset, repo_name, token):
    dataset.push_to_hub(
        repo_name,
        token=token,
        private=False
    )

    print(f"Dataset uploaded successfully to: https://huggingface.co/datasets/{repo_name}")

upload_dataset(dataset, DATASET_REPO, os.environ.get("HF_TOKEN"))

Uploading the dataset shards:   0%|          | 0/1 [00:00<?, ? shards/s]

Creating parquet from Arrow format:   0%|          | 0/1 [00:00<?, ?ba/s]

Processing Files (0 / 0)      : |          |  0.00B /  0.00B            

New Data Upload               : |          |  0.00B /  0.00B            

                              : 100%|##########| 41.2kB / 41.2kB            

...

Creating parquet from Arrow format:   0%|          | 0/1 [00:00<?, ?ba/s]

Processing Files (0 / 0)      : |          |  0.00B /  0.00B            

New Data Upload               : |          |  0.00B /  0.00B            

                              : 100%|##########| 5.79kB / 5.79kB            

Dataset uploaded successfully to: https://huggingface.co/datasets/jujuloaiza/profilerchatbot


## Create the dataset card

In [ ]:
from huggingface_hub import DatasetCard

card_content = f"""---
---
pretty_name: "Profiler Dataset"
license: mit
---

# Profiler Chatbot Dataset

This is a dataset made of examples of generated chat messages from gamers.
"""

card = DatasetCard(card_content)
card.save(f"./README.md")

## Upload the dataset card

In [ ]:
from huggingface_hub import HfApi

api = HfApi()
api.upload_file(
    path_or_fileobj="./README.md",
    path_in_repo="README.md",
    repo_id=DATASET_REPO,
    repo_type="dataset",
)

CommitInfo(commit_url='https://huggingface.co/datasets/jujuloaiza/profilerchatbot/commit/4d65c4dc348ceb7b41c4c4585f88c5650d27b351', commit_message='Upload README.md with huggingface_hub', commit_description='', oid='4d65c4dc348ceb7b41c4c4585f88c5650d27b351', pr_url=None, repo_url=RepoUrl('https://huggingface.co/datasets/jujuloaiza/profilerchatbot', endpoint='https://huggingface.co', repo_type='dataset', repo_id='jujuloaiza/profilerchatbot'), pr_revision=None, pr_num=None)